In [228]:
import numpy as np
from numpy.linalg import norm
import math, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [229]:
#spatial funcs for WoS

def closestPoint(v, s):
    u = s[1] - s[0]
    rate = max(0, min(u.dot(v - s[0])/u.dot(u), 1))
    return (s[0] + rate*(s[1] - s[0]))

def shortestDistance(v, segments):
    r = float("inf")
    for s in segments:
        u = closestPoint(v, s)
        if norm(v - u) < r:
            r = norm(v - u)
    return r

In [230]:
def solver(v, segments, g, eps = 0.01, nWalks = 10, maxSteps = 16):
    vWalks = 0   
    sumEst = 0
    for i in range(0, nWalks):
        x0 = v
        for step in range(0, maxSteps): 
            #r = min([shortestDistance(x0, segments), maxR])
            r = shortestDistance(x0, segments)
            if r < eps: 
                sumEst += g(x0)
                vWalks += 1
                #print("walk: " + str(vWalks) + " hit " + str(x0) + " boundary " + str(g(x0)))
                break
            theta = random.uniform(0, 2*math.pi)
            x0 = x0 + np.array([r * math.cos(theta), r * math.sin(theta)])
    if vWalks == 0:
        return 0
    return sumEst/vWalks

In [231]:
# set up the problem
segments = [np.array([[-1, -1], [-1, 1]]), np.array([[-1, -1], [1, -1]]), np.array([[1, -1], [1, 1]]), np.array([[-1, 1], [1, 1]])]
#v = np.array([0, 0])
def boundary(v):
    return v[0]*v[1]

In [232]:
#####evenly distributed start places
#target = []
#width = range(1, 220)
#height = range(1, 220)
#for i in width:
#    for j in height:
#        target.append(np.array([i/110 - 1, j/110 - 1]))

#print(len(target))

###### pure random start places
target = np.random.rand(10000, 2)
target = 2* target - 1

#####QMC start places
#import scipy.stats as stats
#qmcGenerator = stats.qmc.Sobol(d=2)
#target = qmcGenerator.random(n=40000)
#target = 2* target - 1

In [233]:
#target = np.random.rand(10000, 2)
#target = 2* target - 1

In [234]:
result = []
for v in target:
    result.append(solver(v, segments, boundary))

print(result[100])

KeyboardInterrupt: 

In [ ]:
print(result[2024])
print(target[2024])

In [ ]:
target1 = np.array(target).reshape(-1, 2).astype("float32")
result1 = np.array(result)
result1.shape

In [ ]:
YNNmodel = keras.Sequential( # a list of fully connected layers
    [
        keras.Input(shape = (2)), #not necessary, but for model.summary
        keras.layers.Dense(32, activation = 'relu'),
        keras.layers.Dense(64, activation = 'relu'),
        keras.layers.Dense(128, activation = 'relu'),
        keras.layers.Dense(1), #no softmax activation
    ]
)
print(YNNmodel.summary())


In [ ]:
YNNmodel.compile( # netwrok configurations
    loss = keras.losses.MeanSquaredError(), # do the output softmax
    optimizer = keras.optimizers.Adam(lr = 1e-4),
    #metrics = ["accuracy"],
)

YNNmodel.fit(target1, result1, batch_size = 256, epochs = 50, verbose = 2) # concrete training of the network

In [ ]:
YNNmodel(np.array([-1, 1]).reshape(1, 2))

In [ ]:
fig = plt.figure(figsize = (8, 8))
ax = fig.add_subplot(projection='3d')
X = target[:, 0].reshape(100, 100)
Y = target[:, 1].reshape(100, 100)
Z1 = X*Y

# Plot the 3D surface
#ax.plot_surface(X, Y, Z, edgecolor='royalblue', lw=0.5, rstride=8, cstride=8,
#                alpha=0.3)
ax.plot_surface(X, Y, Z1, edgecolor='red', lw=0.5, rstride=8, cstride=8,
                alpha=0.3)
ax.plot_surface(X, Y, result1.reshape(100, 100), edgecolor='royalblue', lw=0.5, rstride=8, cstride=8,
                alpha=0.3)

ax.set(xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1),
       xlabel='x', ylabel='y', zlabel = 'u(x, y)')
ax.set_box_aspect(None, zoom=0.85)
ax.set_title("Fig1: Original WoS for 2D Laplace", y=-0.01)
#plt.show()
plt.savefig('WoSorgin2Dlaplace.png', bbox_inches='tight')

In [ ]:
target = []
width = range(1, 100)
height = range(1, 100)
for i in width:
    for j in height:
        target.append(np.array([i/50 - 1, j/50 - 1]))
target1 = np.array(target).reshape(-1, 2).astype("float32")
X = target1[:, 0]
Y = target1[:, 1]
Z = []
for i in target1:
        Z.append(YNNmodel(np.array(i).reshape([1, 2]).astype("float32")))

X = np.array(X).reshape(99, 99)
Y = np.array(Y).reshape(99, 99)
Z = np.array(Z).reshape(99, 99)

In [ ]:
Z1 = X*Y
print(Z1.shape)

import matplotlib.pyplot as plt
fig = plt.figure(figsize = (8, 8))
ax = fig.add_subplot(projection='3d')

# Plot the 3D surface
ax.plot_surface(X, Y, Z, edgecolor='royalblue', lw=0.5, rstride=8, cstride=8,
                alpha=0.3)
ax.plot_surface(X, Y, Z1, edgecolor='red', lw=0.5, rstride=8, cstride=8,
                alpha=0.3)
#ax.plot_surface(X, Y, Z2, edgecolor='green', lw=0.5, rstride=8, cstride=8,
#                alpha=0.3)

ax.set(xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1),
       xlabel='x', ylabel='y', zlabel='u(x, y)')
ax.set_box_aspect(None, zoom=0.85)
ax.set_title("Fig2: NN trained by WoS results directly, for 2D Laplace", y=-0.01)
#plt.show()
plt.savefig('WoSsmoothed2Dlaplace.png', bbox_inches='tight')